In [35]:
import pandas as pd
import numpy as np

# RECARGA DESDE CERO (Asegúrate de que los nombres de archivo sean correctos)
# No uses .head() aquí, queremos todo el dataset
try:
    hey_clientes = pd.read_csv('hey_clientes.csv')
    hey_transacciones = pd.read_csv('hey_transacciones.csv')
    hey_productos = pd.read_csv('hey_productos.csv')
    conversaciones = pd.read_csv('dataset_conversaciones/dataset_50k_anonymized_cleaned.csv')
    print("Archivos cargados nuevamente.")
except:
    print("Revisa los nombres de tus archivos .csv")

# ARREGLAR IDs (Usando .loc para evitar el SettingWithCopyWarning)
hey_clientes.loc[:, 'user_id'] = hey_clientes['user_id'].astype(str).str.strip()
hey_transacciones.loc[:, 'user_id'] = hey_transacciones['user_id'].astype(str).str.strip()
hey_productos.loc[:, 'user_id'] = hey_productos['user_id'].astype(str).str.strip()
conversaciones.loc[:, 'user_id'] = conversaciones['user_id'].astype(str).str.strip()

# VERIFICACIÓN CRÍTICA
print(f"--- VERIFICACIÓN DE INTEGRIDAD ---")
print(f"Total filas en Clientes: {len(hey_clientes)}")
print(f"IDs únicos en Clientes: {hey_clientes['user_id'].nunique()}")
print(f"IDs únicos en Transacciones: {hey_transacciones['user_id'].nunique()}")

# Si aquí te sigue saliendo '5', el problema está en tu archivo original 'clientes_limpio.csv'

Archivos cargados nuevamente.
--- VERIFICACIÓN DE INTEGRIDAD ---
Total filas en Clientes: 15025
IDs únicos en Clientes: 15025
IDs únicos en Transacciones: 15025


In [32]:

import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go

# 1. CARGA DE DATOS
clientes = pd.read_csv('hey_clientes.csv')
transacciones = pd.read_csv('hey_transacciones.csv')
productos = pd.read_csv('hey_productos.csv')

# Cargar conversaciones desde el subdirectorio
conversaciones = pd.read_csv('dataset_conversaciones/dataset_50k_anonymized_cleaned.csv')

print("Datos cargados exitosamente:")
print(f"  - Clientes: {len(clientes):,} registros")
print(f"  - Transacciones: {len(transacciones):,} registros")
print(f"  - Productos: {len(productos):,} registros")
print(f"  - Conversaciones: {len(conversaciones):,} registros")

hey_clientes = clientes.head()
hey_clientes

hey_transacciones = transacciones.head()
hey_transacciones

hey_productos = productos.head()
hey_productos


#mostrar titulos de los datos cargados
print("Columnas en hey_clientes:", hey_clientes.columns.tolist())
print("Columnas en hey_transacciones:", hey_transacciones.columns.tolist())
print("Columnas en hey_productos:", productos.columns.tolist())
print("Columnas en conversaciones:", conversaciones.columns.tolist())




Datos cargados exitosamente:
  - Clientes: 15,025 registros
  - Transacciones: 802,384 registros
  - Productos: 38,909 registros
  - Conversaciones: 49,976 registros
Columnas en hey_clientes: ['user_id', 'edad', 'sexo', 'estado', 'ciudad', 'nivel_educativo', 'ocupacion', 'ingreso_mensual_mxn', 'antiguedad_dias', 'es_hey_pro', 'nomina_domiciliada', 'canal_apertura', 'score_buro', 'dias_desde_ultimo_login', 'preferencia_canal', 'satisfaccion_1_10', 'recibe_remesas', 'usa_hey_shop', 'idioma_preferido', 'tiene_seguro', 'num_productos_activos', 'patron_uso_atipico']
Columnas en hey_transacciones: ['transaccion_id', 'user_id', 'producto_id', 'fecha_hora', 'tipo_operacion', 'canal', 'monto', 'comercio_nombre', 'categoria_mcc', 'ciudad_transaccion', 'estatus', 'motivo_no_procesada', 'intento_numero', 'meses_diferidos', 'cashback_generado', 'descripcion_libre', 'hora_del_dia', 'dia_semana', 'es_internacional', 'dispositivo', 'patron_uso_atipico', 'es_dato_sintetico']
Columnas en hey_productos: 

In [36]:
# 1. Forzar que todos los user_id sean STRING y sin espacios
hey_clientes['user_id'] = hey_clientes['user_id'].astype(str).str.strip()
hey_transacciones['user_id'] = hey_transacciones['user_id'].astype(str).str.strip()
hey_productos['user_id'] = hey_productos['user_id'].astype(str).str.strip()
conversaciones['user_id'] = conversaciones['user_id'].astype(str).str.strip()

print(f"IDs normalizados. Verificando conteos únicos:")
print(f"- Clientes: {hey_clientes['user_id'].nunique()}")
print(f"- Clientes en Transacciones: {hey_transacciones['user_id'].nunique()}")

# 2. Re-hacer los resúmenes asegurando que el índice sea el user_id
perfil_trans = hey_transacciones.groupby('user_id').agg({
    'monto': 'sum',
    'estatus': lambda x: (x != 'Aprobada').sum(),
    'categoria_mcc': lambda x: x.mode()[0] if not x.empty else 'Sin Datos'
}).reset_index()

perfil_prod = hey_productos.groupby('user_id').size().reset_index(name='cantidad_productos')

perfil_conv = conversaciones.groupby('user_id').size().reset_index(name='num_conversaciones')

# 3. UNIÓN IZQUIERDA (LEFT JOIN) - La clave es 'left' para no perder clientes
master_df = hey_clientes.copy() # Empezamos con todos los clientes
master_df = master_df.merge(perfil_trans, on='user_id', how='left')
master_df = master_df.merge(perfil_prod, on='user_id', how='left')
master_df = master_df.merge(perfil_conv, on='user_id', how='left')

# 4. Llenar vacíos (quienes no tienen transacciones/productos)
master_df.fillna({'monto': 0, 'estatus': 0, 'cantidad_productos': 0, 'num_conversaciones': 0}, inplace=True)

print(f"--- RESULTADO FINAL ---")
print(f"Total de clientes en Master: {master_df.shape[0]}")

IDs normalizados. Verificando conteos únicos:
- Clientes: 15025
- Clientes en Transacciones: 15025
--- RESULTADO FINAL ---
Total de clientes en Master: 15025


In [37]:
import pandas as pd
import numpy as np
import plotly.express as px

# --- 1. PROCESAR TRANSACCIONES (La base del comportamiento) ---
# Calculamos: Gasto total, Rechazos, y la Categoría donde más gasta
perfil_trans = hey_transacciones.groupby('user_id').agg({
    'monto': ['sum', 'mean'],
    'estatus': lambda x: (x != 'Aprobada').sum(), # Cuenta fallidas
    'categoria_mcc': lambda x: x.mode()[0] if not x.empty else 'Sin Categoría',
    'cashback_generado': 'sum'
})
perfil_trans.columns = ['gasto_total', 'ticket_promedio', 'num_rechazos', 'categoria_top', 'total_cashback']
perfil_trans = perfil_trans.reset_index()

# --- 2. PROCESAR PRODUCTOS ---
# Calculamos: Cuántos productos tiene y su deuda/saldo total
perfil_prod = hey_productos.groupby('user_id').agg({
    'tipo_producto': 'count',
    'saldo_actual': 'sum',
    'utilizacion_pct': 'mean'
}).rename(columns={'tipo_producto': 'cantidad_productos'}).reset_index()

# --- 3. PROCESAR CONVERSACIONES ---
# Calculamos: Frecuencia de uso del bot y canal preferido
perfil_conv = conversaciones.groupby('user_id').agg({
    'conv_id': 'nunique',
    'channel_source': lambda x: x.mode()[0] if not x.empty else 1
}).rename(columns={'conv_id': 'num_conversaciones', 'channel_source': 'canal_bot_fav'}).reset_index()

# --- 4. EL GRAN MASTER JOIN (Visión 360) ---
master_df = hey_clientes.merge(perfil_trans, on='user_id', how='left') \
                        .merge(perfil_prod, on='user_id', how='left') \
                        .merge(perfil_conv, on='user_id', how='left')

# Limpieza final: Llenar ceros para los que no tienen actividad
cols_fill_zero = ['gasto_total', 'num_rechazos', 'cantidad_productos', 'num_conversaciones', 'total_cashback']
master_df[cols_fill_zero] = master_df[cols_fill_zero].fillna(0)

print(f"Tabla Maestra lista: {master_df.shape[0]} clientes procesados.")

Tabla Maestra lista: 15025 clientes procesados.


In [38]:
fig1 = px.scatter(master_df, 
                 x="num_rechazos", 
                 y="satisfaccion_1_10",
                 size="gasto_total", 
                 color="es_hey_pro",
                 hover_data=['user_id', 'categoria_top'],
                 title="<b>Insight 1:</b> Fricción Transaccional vs Satisfacción del Cliente",
                 labels={'num_rechazos': 'Transacciones Fallidas', 'satisfaccion_1_10': 'Nivel de Satisfacción'},
                 template="plotly_white", color_discrete_sequence=["#FF0000", "#00FF00"])
fig1.show()

In [39]:
fig2 = px.box(master_df, 
             x="ocupacion", 
             y="total_cashback", 
             color="tiene_seguro",
             title="<b>Insight 2:</b> Generación de Cashback por Ocupación y Tenencia de Seguro",
             notched=True)
fig2.update_layout(xaxis={'categoryorder':'total descending'})
fig2.show()

In [40]:
fig3 = px.histogram(master_df, 
                   x="score_buro", 
                   color="canal_bot_fav", 
                   marginal="rug",
                   title="<b>Insight 3:</b> Perfil de Riesgo (Buró) vs Canal de Interacción",
                   labels={'canal_bot_fav': 'Canal (1=Escrito, 2=Voz)'},
                   barmode="overlay")
fig3.show()

In [41]:
def motor_decisiones(row):
    # Lógica de IA Proactiva
    if row['num_rechazos'] > 3 and row['satisfaccion_1_10'] < 5:
        return "RETENCIÓN: Llamada inmediata - Problemas Técnicos"
    elif row['gasto_total'] > 10000 and row['tiene_seguro'] == 0:
        return "VENTA: Oferta de Seguro basada en alto gasto"
    elif row['score_buro'] > 700 and row['cantidad_productos'] < 2:
        return "CRECIMIENTO: Ofrecer Hey Pro (Perfil Premium)"
    elif row['num_conversaciones'] > 5 and row['canal_bot_fav'] == 2:
        return "OPTIMIZACIÓN: Migrar a Bot de Voz Inteligente"
    else:
        return "MANTENER: Comunicación estándar"

master_df['Accion_Sugerida_IA'] = master_df.apply(motor_decisiones, axis=1)

# Visualizar el impacto de la estrategia
resumen_estrategia = master_df['Accion_Sugerida_IA'].value_counts().reset_index()
fig4 = px.pie(resumen_estrategia, values='count', names='Accion_Sugerida_IA', 
             title="<b>Estrategia Final:</b> Distribución de Acciones Proactivas del Motor de IA")
fig4.show()

In [42]:
# 1. Agrupar transacciones por usuario
resumen_trans = hey_transacciones.groupby('user_id').agg({
    'monto': ['sum', 'mean', 'count'],
    'estatus': lambda x: (x != 'Aprobada').sum()
})
resumen_trans.columns = ['gasto_total', 'ticket_promedio', 'num_transacciones', 'num_rechazos']
resumen_trans = resumen_trans.reset_index()

# 2. Agrupar conversaciones (Descubrimiento de actividad)
resumen_conv = conversaciones.groupby('user_id').size().reset_index(name='num_conversaciones')

# 3. Unión Maestra
df_master = hey_clientes.merge(resumen_trans, on='user_id', how='left') \
                        .merge(resumen_conv, on='user_id', how='left')

# Llenar nulos con 0
df_master[['gasto_total', 'num_rechazos', 'num_conversaciones']] = df_master[['gasto_total', 'num_rechazos', 'num_conversaciones']].fillna(0)

In [43]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Seleccionamos variables clave para descubrir patrones
features = ['edad', 'ingreso_mensual_mxn', 'score_buro', 'gasto_total', 'num_rechazos']
df_cluster = df_master[features].fillna(0)

# Escalamos
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)

# Aplicamos K-Means
kmeans = KMeans(n_clusters=4, random_state=42)
df_master['segmento_ia'] = kmeans.fit_predict(df_scaled)

# Ver qué descubrimos
print("Distribución de segmentos descubiertos:")
print(df_master['segmento_ia'].value_counts())

Distribución de segmentos descubiertos:
segmento_ia
0    4199
1    4040
2    3875
3    2911
Name: count, dtype: int64


In [44]:
# Palabras clave de necesidades
necesidades = {
    'seguros': ['seguro', 'proteccion', 'siniestro', 'poliza'],
    'inversiones': ['invertir', 'rendimiento', 'tasa', 'pagare', 'ahorro'],
    'tarjetas': ['limite', 'anualidad', 'credito', 'platino']
}

def detectar_necesidad(text):
    if not isinstance(text, str): return "Ninguna"
    text = text.lower()
    for categoria, keywords in necesidades.items():
        if any(word in text for word in keywords):
            return categoria
    return "General"

conversaciones['necesidad_detectada'] = conversaciones['input'].apply(detectar_necesidad)

In [45]:
import plotly.express as px

fig_final = px.scatter(df_master, 
                      x="ingreso_mensual_mxn", 
                      y="gasto_total", 
                      color="segmento_ia",
                      size="num_conversaciones",
                      hover_data=['user_id', 'nivel_educativo'],
                      title="<b>Motor de Inteligencia:</b> Segmentos de Clientes Descubiertos",
                      labels={'segmento_ia': 'Segmento IA', 'ingreso_mensual_mxn': 'Ingresos'})
fig_final.show()

In [46]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- PRE-PROCESAMIENTO PARA INSIGHTS ---
# 1. Agrupamos Transacciones (Comportamiento de Gasto)
trans_agg = hey_transacciones.groupby('user_id').agg(
    gasto_total=('monto', 'sum'),
    rechazos=('estatus', lambda x: (x != 'Aprobada').sum()),
    categoria_frecuente=('categoria_mcc', lambda x: x.mode()[0] if not x.empty else 'N/A')
).reset_index()

# 2. Agrupamos Conversaciones (Engagement)
conv_agg = conversaciones.groupby('user_id').agg(
    total_chats=('user_id', 'count'),
    canal_pref=('channel_source', lambda x: 'Voz' if x.mode()[0] == 2 else 'Texto')
).reset_index()

# 3. Join Final (Master Dashboard Data)
db_df = hey_clientes.merge(trans_agg, on='user_id', how='left') \
                    .merge(conv_agg, on='user_id', how='left')
db_df.fillna({'gasto_total':0, 'rechazos':0, 'total_chats':0, 'canal_pref':'N/A'}, inplace=True)

In [48]:
# Configuramos el layout del Dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "1. El Cuadrante de Fricción (Gasto vs Rechazos)", 
        "2. Preferencia de Canal por Nivel Socioeconómico",
        "3. Mapa de Calor: Gasto por Categoría y Estatus Pro",
        "4. Embudo de Engagement: ¿Quiénes usan más al asistente Havi?"
    ),
    specs=[[{"type": "scatter"}, {"type": "box"}],
           [{"type": "treemap"}, {"type": "histogram"}]]
)

# --- INSIGHT 1: FRICCIÓN (VIPs EN RIESGO) ---
fig.add_trace(
    go.Scatter(x=db_df['gasto_total'], y=db_df['rechazos'], mode='markers',
               marker=dict(size=db_df['total_chats']*2, color=db_df['ingreso_mensual_mxn'], colorscale='Viridis'),
               text=db_df['user_id'], name="Clientes"),
    row=1, col=1
)

# --- INSIGHT 2: SEGMENTACIÓN DE CANAL ---
fig.add_trace(
    go.Box(x=db_df['canal_pref'], y=db_df['ingreso_mensual_mxn'], name="Ingresos por Canal"),
    row=1, col=2
)

# --- INSIGHT 3: DEMANDA DE CATEGORÍAS (TREEMAP) ---
# Mostramos dónde está el dinero según el estatus de la cuenta
fig.add_trace(
    go.Treemap(
        labels=db_df['categoria_frecuente'],
        parents=[""] * len(db_df),
        values=db_df['gasto_total'],
        branchvalues="total"
    ),
    row=2, col=1
)

# --- INSIGHT 4: ADOPCIÓN DE LA IA ---
fig.add_trace(
    go.Histogram(x=db_df['total_chats'], nbinsx=20, name="Frecuencia de Uso Havi"),
    row=2, col=2
)

# Estética del Dashboard
fig.update_layout(height=800, title_text="<b>Dashboard de Insights: Motor de Inteligencia Havi</b>", 
                  showlegend=False, template="plotly_dark")
fig.show()

In [ ]:
from flask import Flask, request, jsonify
import pandas as pd

app = Flask(__name__)

# Supongamos que df_master es la tabla que ya limpiamos y unimos
# df_master = pd.read_csv('master_consolidado.csv') 

def generar_pregunta_proactiva(user_id):
    # 1. Extraer el perfil del cliente desde nuestra tabla de insights
    cliente = df_master[df_master['user_id'] == user_id]
    
    if cliente.empty:
        return "¡Hola! Soy Havi, ¿en qué puedo ayudarte hoy?"

    # Sacamos variables clave
    nombre = "Cliente" # O si tienes la columna nombre
    gasto = cliente['gasto_total'].values[0]
    rechazos = cliente['num_rechazos'].values[0]
    giro = cliente['categoria_top'].values[0]
    es_pro = cliente['es_hey_pro'].values[0]
    cashback = cliente['total_cashback'].values[0]

    # 2. LÓGICA DE NEGOCIO (El Motor de Inteligencia)
    # Caso A: Fricción detectada (Insight de Rechazos)
    if rechazos > 0:
        return f"Hola, noté que tuviste {int(rechazos)} inconvenientes con tus últimos pagos. ¿Quieres que revisemos los límites de tu tarjeta para que no vuelva a pasar?"

    # Caso B: Cross-sell basado en Gasto (Insight de Valor)
    if gasto > 5000 and es_pro == 'No':
        return f"¡Impresionante! Has gastado ${gasto:,.2f} este mes. Si fueras Hey Pro, habrías ganado el doble de cashback. ¿Te gustaría activarlo ahora?"

    # Caso C: Insight de Gastos (Descubrimiento)
    if giro == 'Restaurantes':
        return f"Veo que eres un foodie. ¿Sabías que este fin de semana tienes 5% de descuento adicional en restaurantes con tu tarjeta?"

    # Caso D: Inversión (Insight de Cashback)
    if cashback > 100:
        return f"Tienes ${cashback:,.2f} acumulados de cashback sin usar. ¿Te gustaría invertirlos para que generen rendimientos?"

    return "¿En qué más puedo asesorarte hoy para mejorar tus finanzas?"

@app.route('/get_bot_question', methods=['POST'])
def bot_question():
    data = request.json
    uid = data.get('user_id')
    pregunta = generar_pregunta_proactiva(uid)
    return jsonify({"pregunta": pregunta})

if __name__ == '__main__':
    app.run(debug=True)